# 🔧 Fine-Tuning Exploration: Therapy Companion

**Goal**: Test LoRA on synthetic/personal data to improve parent/emotion responses.

Uses your `ModelManager`. Creates 100 synthetic therapy examples, trains LoRA, tests before/after.

In [1]:
# Setup
import logging
import json
from pathlib import Path

logging.basicConfig(level=logging.INFO)

from therapy_ai.core.model_manager import ModelManager
from therapy_ai.core.backend import BackendFactory, HardwareBackend

# Check backend supports fine-tuning
info = BackendFactory.detect()
if info.backend != HardwareBackend.MLX:
    raise RuntimeError(f"Fine-tuning needs MLX. Detected: {info}")

print(f"✅ Fine-tuning ready on {info}")

# Load base model
manager = ModelManager.from_config()
base_model = manager.load()
print("Base model loaded")

INFO:therapy_ai.core.backend:Backend detected: [MLX] Apple M4 | 16.0 GB | ✓ fine-tuning


✅ Fine-tuning ready on [MLX] Apple M4 | 16.0 GB | ✓ fine-tuning


FileNotFoundError: [Errno 2] No such file or directory: 'config/default_config.yaml'

In [ ]:
# Generate synthetic therapy dataset (your use case)
import random

PARENT_TOPICS = [
    "resentment", "guilt", "criticism", "expectations", "abandonment", "control",
    "approval", "comparison", "emotional distance", "high standards"
]
ROMANCE_PATTERNS = [
    "choosing familiar types", "fear of intimacy", "self-sabotage", "trust issues",
    "people-pleasing", "avoidance", "high standards", "fear of rejection"
]

def generate_synthetic_dataset(n_samples=100):
    dataset = []
    for i in range(n_samples):
        topic = random.choice([PARENT_TOPICS, ROMANCE_PATTERNS])[random.randint(0,6)]
        user_input = f"I feel conflicted about {topic} in my {random.choice(['parent', 'romantic']) } relationships."
        
        # Ideal therapy response structure
        assistant = random.choice([
            f"That {topic} sounds heavy to carry. What comes up first when you think about it?",
            f"Conflict around {topic} often has deep roots. Where do you feel it most in your body?",
            f"'{topic}' — that's a powerful word. Can you tell me more about what it means to you?",
            f"Many carry {topic} from family patterns. How does it show up for you day-to-day?"
        ])
        
        dataset.append({
            "messages": [
                {"role": "system", "content": "Empathetic reflective companion. Ask one question."},
                {"role": "user", "content": user_input},
                {"role": "assistant", "content": assistant}
            ]
        })
    return dataset

dataset = generate_synthetic_dataset(100)
print(f"Generated {len(dataset)} therapy examples")
(Path("data/finetune/train.jsonl").write_text(
    "\n".join(json.dumps(ex, separators=(',', ':')) for ex in dataset)
))
print("✅ Saved data/finetune/train.jsonl")

In [ ]:
# Test BASELINE performance
from mlx_lm import generate

test_prompt = dataset[0]["messages"][1:][0]["content"]
print("=== BASELINE (pre-fine-tune) ===")
baseline_response = generate(
    base_model.model, base_model.tokenizer,
    prompt=base_model.tokenizer.apply_chat_template([
        {"role": "system", "content": "Reflective companion"},
        {"role": "user", "content": test_prompt}
    ], add_generation_prompt=True),
    max_tokens=200, temp=0.7
)
print(baseline_response)

In [ ]:
# Fine-tune with mlx_tune LoRA (15min on M4)
from mlx_tune import SFTTrainer, TrainingArguments

trainer_args = TrainingArguments(
    output_dir="data/adapters/therapy-lora",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    logging_steps=10,
    save_steps=50,
    max_steps=100,  # Quick test
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=base_model.model,
    tokenizer=base_model.tokenizer,
    train_dataset=dataset,
    args=trainer_args,
    packing=True,
)

print("🚀 Starting fine-tuning...")
trainer.train()
trainer.save_model()
print("✅ LoRA saved: data/adapters/therapy-lora")

In [ ]:
# Test FINE-TUNED performance
print("=== FINE-TUNED ===")
manager.attach_lora("data/adapters/therapy-lora")

finetuned_response = generate(
    manager.loaded.model, manager.loaded.tokenizer,
    prompt=manager.tokenizer.apply_chat_template([
        {"role": "system", "content": "Reflective companion"},
        {"role": "user", "content": test_prompt}
    ], add_generation_prompt=True),
    max_tokens=200, temp=0.7
)
print(finetuned_response)

# Compare
print("\n=== IMPROVEMENT? ===")
print("Baseline:", baseline_response[:100] + "...")
print("Fine-tuned:", finetuned_response[:100] + "...")

In [ ]:
# Cleanup
manager.unload()
print("Done. Update config adapter_path to use in app!")